<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_01_logistic_regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_01 - T2 SEQ2ONE - Baseline**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-21 17:50:58,182 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

Mounted at /content/drive


2026-04-21 17:51:25,107 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-21 17:51:47,960 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-21 17:51:47,962 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-21 17:51:47,963 | INFO | Configuración de experimento cargada
2026-04-21 17:51:47,963 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-21 17:51:47,964 | INFO | Window sizes: [30]


In [8]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-21 17:54:40,979 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-21 17:54:40,985 | INFO | Windows OK      : 9
2026-04-21 17:54:40,986 | INFO | Windows missing : 0
2026-04-21 17:54:40,986 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-21 17:54:40,987 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [9]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [10]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [13]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [14]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [15]:
bundles_L30 = create_bundles(window_size=30)

2026-04-21 18:01:54,776 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-21 18:01:54,777 | INFO | X shape: (32147, 30, 8) | y shape: (32147,)
2026-04-21 18:01:55,634 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-21 18:01:55,634 | INFO | X shape: (6882, 30, 8) | y shape: (6882,)
2026-04-21 18:01:56,442 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-21 18:01:56,443 | INFO | X shape: (6913, 30, 8) | y shape: (6913,)
2026-04-21 18:01:58,268 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-21 18:01:58,269 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 8) | valid=(6882, 30, 8) | test=(6913, 30, 8)
2026-04-21 18:02:00,248 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-21 18:02:00,249 | INFO | X shape: (32147, 30, 8) | y shape: (32147,)
2026-04-21 18:02:01,037 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-21 18:02:01,038 | INFO | X shape: (6882, 30, 8) | y shape: (6882,)
2026-04-21 18:02:01,964 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 8) (32147,)
Valid : (6882, 30, 8) (6882,)
Test  : (6913, 30, 8) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 8) (32147,)
Valid : (6882, 30, 8) (6882,)
Test  : (6913, 30, 8) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 8) (32147,)
Valid : (6882, 30, 8) (6882,)
Test  : (6913, 30, 8) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




### **4.5. Preparación de inputs según el tipo de modelo**

In [ ]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [16]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [17]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 8) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 8) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 8) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 8) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 8) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 8) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 8) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 8) | y=(6882,) | mode=3d | n_classes=3 | c

In [ ]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **6. Métricas de clasificación T2**

In [ ]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-07 23:45:19,553 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [ ]:
# ================================
# Carga de métricas (si existen)
# ================================

def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.
    """

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(f"No existen métricas previas para: {name}")
    return pd.DataFrame()

In [ ]:
# ================================
# Guardado de métricas
# ================================

def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path

Ejemplo de uso:

```python
df_metrics = load_classification_metrics_if_exists(name="lstm_valid")

df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)

save_classification_metrics(df_metrics, name="lstm_valid")
```



## **8. Gestión de dispositivo y memoria**

In [ ]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [ ]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-07 23:45:22,506 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo**

## **10.1. Función unitaria por bundle**

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np


def run_logistic_for_bundle_seq2one(
    bundle,
    *,
    multi_class="multinomial",
    max_iter=1000,
    C=1.0,
    random_state=42,
    input_mode="2d_flat",
    class_weight=None,
    verbose=False,
):
    """
    Ejecuta Logistic Regression para un bundle seq2one.

    Parámetros clave
    ----------------
    class_weight : None | "balanced" | dict
        - None       -> entrenamiento natural con clases desbalanceadas
        - "balanced" -> pondera automáticamente según frecuencia inversa
        - dict       -> pesos definidos manualmente, ej: {-1: 2.0, 0: 1.0, 1: 2.0}

    Retorna
    -------
    dict con modelo y predicciones sobre VALID y TEST
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    X_test  = bundle["test"]["X"]

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)
    X_test_model  = prepare_X_for_model(X_test,  input_mode=input_mode)

    # =========================
    # 3. MODELO
    # =========================
    model = LogisticRegression(
        multi_class=multi_class,
        max_iter=max_iter,
        C=C,
        random_state=random_state,
        n_jobs=-1,
        class_weight=class_weight,
    )

    # =========================
    # 4. TRAIN
    # =========================
    model.fit(X_train_model, y_train)

    # =========================
    # 5. PREDICT
    # =========================
    y_pred_valid = model.predict(X_valid_model)
    y_pred_test  = model.predict(X_test_model)

    y_proba_valid = model.predict_proba(X_valid_model)
    y_proba_test  = model.predict_proba(X_test_model)

    return {
        "model": model,
        "class_weight": class_weight,
        "y_pred_valid": y_pred_valid,
        "y_pred_test": y_pred_test,
        "y_proba_valid": y_proba_valid,
        "y_proba_test": y_proba_test,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [ ]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_logistic_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    input_mode: str = "2d_flat",
    class_weight=None,  # 👈 NUEVO
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa Logistic Regression para uno o varios bundles seq2one y
    retorna un DataFrame consolidado.

    Incluye soporte para clases desbalanceadas o balanceadas.

    class_weight:
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática
    """

    # --------------------------------------------------
    # 1) Normalizar entrada a lista
    # --------------------------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    # --------------------------------------------------
    # 2) Validar split
    # --------------------------------------------------
    if split not in ("valid", "test"):
        raise ValueError("split debe ser 'valid' o 'test'")

    rows = []

    # --------------------------------------------------
    # 3) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 4) Entrenar + predecir
        # ----------------------------------------------
        preds = run_logistic_for_bundle_seq2one(
            bundle,
            multi_class=multi_class,
            max_iter=max_iter,
            C=C,
            random_state=random_state,
            input_mode=input_mode,
            class_weight=class_weight,  # 👈 NUEVO
            verbose=False,
        )

        # ----------------------------------------------
        # 5) Seleccionar y_true / y_pred del split
        # ----------------------------------------------
        y_true = bundle[split]["y"]
        y_pred_key = f"y_pred_{split}"

        if y_pred_key not in preds:
            raise KeyError(
                f"No existe '{y_pred_key}' en la salida de "
                f"run_logistic_for_bundle_seq2one"
            )

        y_pred = preds[y_pred_key]

        # ----------------------------------------------
        # 6) Métricas de clasificación
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split=split,
            target=target,
            labels=[-1, 0, 1],
        )

        # ----------------------------------------------
        # 7) A DataFrame
        # ----------------------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split=split,
            window_size=window_size,
            target=target,
        )

        df_row["horizon_min"] = horizon

        # 👇 MUY IMPORTANTE: distinguir tipo de entrenamiento
        df_row["class_weight_mode"] = (
            "balanced" if class_weight == "balanced" else "none"
        )

        rows.append(df_row)

    # --------------------------------------------------
    # 8) Consolidar salida
    # --------------------------------------------------
    return pd.concat(rows, ignore_index=True)

Cómo se usaría, para ambos bundles de una ventana:

```python
df_valid = eval_logistic_bundles(
    [bundle_t2_90, bundle_t2_120],
    split="valid",
    model_name="logistic_regression",
    verbose=True,
)

df_test = eval_logistic_bundles(
    [bundle_t2_90, bundle_t2_120],
    split="test",
    model_name="logistic_regression",
    verbose=True,
)
```




## **10.3. Función orquestadora por `window_size`**

In [ ]:
import gc
import pandas as pd


def run_logistic(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    input_mode: str = "2d_flat",
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta Logistic Regression para una sola window_size
    sobre los targets T2:
      - t2_dir_thr_90
      - t2_dir_thr_120

    Retorna un DataFrame consolidado con métricas de VALID y TEST.

    Parámetros
    ----------
    class_weight : None | "balanced" | dict
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática por frecuencia inversa
        - dict       -> pesos definidos manualmente
    """

    size = int(window_size)

    bundle_t2_90 = bundle_t2_120 = None
    bundles_t2 = None
    df_out = None

    # nombre visible del experimento
    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"LOGISTIC REGRESSION | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"class_weight = {class_weight}")

        # --------------------------------------------------
        # 2) Construcción de bundles para esta ventana
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['t2_dir_thr_90', 't2_dir_thr_120']")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90, bundle_t2_120]

        # --------------------------------------------------
        # 3) Evaluación por split
        # --------------------------------------------------
        dfs = []

        for split in ["valid", "test"]:
            if verbose:
                print(
                    f"\n[EVAL] L{size} | split={split} | "
                    f"model={model_name_effective} | T2 | class_weight={class_weight}"
                )

            df_split = eval_logistic_bundles(
                bundles_t2,
                split=split,
                model_name=model_name_effective,
                max_iter=max_iter,
                random_state=random_state,
                C=C,
                multi_class=multi_class,
                input_mode=input_mode,
                class_weight=class_weight,
                verbose=verbose,
            )
            dfs.append(df_split)

        # --------------------------------------------------
        # 4) Consolidación final
        # --------------------------------------------------
        df_out = (
            pd.concat(dfs, ignore_index=True)
            .sort_values(
                ["window_size", "target", "split", "horizon_min", "model"]
            )
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen final
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "split",
                        "target",
                        "model",
                        "horizon_min",
                        "class_weight_mode",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .sort_values(
                    ["split", "target", "model", "horizon_min", "class_weight_mode"]
                )
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()

## **10.4. Función incremental multi-ventana**

In [ ]:
from pathlib import Path
import pandas as pd


def run_logistic_incremental(
    *,
    window_sizes: list[int],
    name: str = "logistic_regression",
    verbose: bool = True,
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    input_mode: str = "2d_flat",
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta Logistic Regression de forma incremental para múltiples window_sizes.

    - Carga histórico si existe
    - Hace SKIP si un window_size ya está completo
    - Corre run_logistic(window_size=L) para los faltantes
    - Agrega resultados nuevos al histórico
    - Guarda usando save_classification_metrics(df_hist, name=name)

    Parámetros
    ----------
    class_weight : None | "balanced" | dict
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática por frecuencia inversa
        - dict       -> pesos definidos manualmente
    """

    # nombre efectivo del experimento
    name_effective = f"{name}_balanced" if class_weight == "balanced" else name

    metrics_dir = DRIVE_DIR / "metrics/classification_metrics"
    metrics_path = metrics_dir / f"classification_{name_effective}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico si existe
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definir completitud esperada por window_size
    # --------------------------------------------------
    expected_targets = {"t2_dir_thr_90", "t2_dir_thr_120"}
    expected_splits = {"valid", "test"}
    expected_models = {name_effective}
    expected_class_weight_mode = {
        "balanced" if class_weight == "balanced" else "none"
    }

    # --------------------------------------------------
    # 3) Iterar por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)

        # ----------------------------------------------
        # Skip robusto por window_size
        # ----------------------------------------------
        if not df_hist.empty:
            dfL = df_hist[df_hist["window_size"] == L]

            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits = set(dfL["split"].unique()) if not dfL.empty else set()
            done_models = set(dfL["model"].unique()) if not dfL.empty else set()

            if "class_weight_mode" in dfL.columns:
                done_class_weight_mode = (
                    set(dfL["class_weight_mode"].unique()) if not dfL.empty else set()
                )
            else:
                done_class_weight_mode = set()

            is_complete = (
                expected_targets.issubset(done_targets)
                and expected_splits.issubset(done_splits)
                and expected_models.issubset(done_models)
                and expected_class_weight_mode.issubset(done_class_weight_mode)
            )

            if is_complete:
                if verbose:
                    print(
                        f"[SKIP] {name_effective} L={L} ya existe completo en Drive"
                    )
                continue

        # ----------------------------------------------
        # Ejecutar Logistic para este window_size
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 80)
            print(f"[RUN] {name_effective} | L={L} | class_weight={class_weight}")
            print("-" * 80)

        df_L = run_logistic(
            window_size=L,
            verbose=verbose,
            model_name=name,
            max_iter=max_iter,
            random_state=random_state,
            C=C,
            multi_class=multi_class,
            input_mode=input_mode,
            class_weight=class_weight,
        )

        # Etiqueta de familia
        df_L["family"] = name_effective

        # ----------------------------------------------
        # Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # Eliminar duplicados por seguridad
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
        ]
        if "class_weight_mode" in df_hist.columns:
            subset_cols.append("class_weight_mode")

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # Guardar histórico actualizado
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name_effective)

    # --------------------------------------------------
    # 4) Retorno final ordenado
    # --------------------------------------------------
    sort_cols = ["window_size", "target", "split", "horizon_min", "model"]
    if "class_weight_mode" in df_hist.columns:
        sort_cols.append("class_weight_mode")

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **10.5. Ejecución final del experimento**

### Sin balanceo de clases

In [ ]:
df_logistic_none = run_logistic_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="logistic_regression",
    class_weight=None,
)

[SKIP] logistic_regression L=30 ya existe completo en Drive
[SKIP] logistic_regression L=60 ya existe completo en Drive
[SKIP] logistic_regression L=90 ya existe completo en Drive
[SKIP] logistic_regression L=120 ya existe completo en Drive
[SKIP] logistic_regression L=180 ya existe completo en Drive


### Con balanceo de clases

In [ ]:
df_logistic_bal = run_logistic_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="logistic_regression",
    class_weight="balanced",
)

[SKIP] logistic_regression_balanced L=30 ya existe completo en Drive
[SKIP] logistic_regression_balanced L=60 ya existe completo en Drive
[SKIP] logistic_regression_balanced L=90 ya existe completo en Drive
[SKIP] logistic_regression_balanced L=120 ya existe completo en Drive
[SKIP] logistic_regression_balanced L=180 ya existe completo en Drive


In [ ]:
df_logistic_none

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,logistic_regression,test,30,t2_dir_thr_120,93990,0.345316,0.270370,0.438369,0.579987,0.383358,0.345316,0.333333,0.011983,120,none,logistic_regression
1,logistic_regression,valid,30,t2_dir_thr_120,93508,0.335323,0.283183,0.604690,0.720698,0.551519,0.335323,0.333333,0.001990,120,none,logistic_regression
2,logistic_regression,test,30,t2_dir_thr_90,93990,0.343799,0.268514,0.436526,0.577668,0.409567,0.343799,0.333333,0.010466,90,none,logistic_regression
3,logistic_regression,valid,30,t2_dir_thr_90,93508,0.335520,0.282303,0.596703,0.714420,0.497953,0.335520,0.333333,0.002187,90,none,logistic_regression
4,logistic_regression,test,60,t2_dir_thr_120,88140,0.344856,0.265729,0.420574,0.564772,0.428477,0.344856,0.333333,0.011523,120,none,logistic_regression
5,logistic_regression,valid,60,t2_dir_thr_120,87688,0.335008,0.279094,0.584491,0.705456,0.546489,0.335008,0.333333,0.001675,120,none,logistic_regression
6,logistic_regression,test,60,t2_dir_thr_90,88140,0.344926,0.266854,0.421225,0.564012,0.444605,0.344926,0.333333,0.011593,90,none,logistic_regression
7,logistic_regression,valid,60,t2_dir_thr_90,87688,0.335449,0.278536,0.576180,0.698830,0.517727,0.335449,0.333333,0.002115,90,none,logistic_regression
8,logistic_regression,test,90,t2_dir_thr_120,82290,0.345404,0.262758,0.403735,0.549642,0.440451,0.345404,0.333333,0.012071,120,none,logistic_regression
9,logistic_regression,valid,90,t2_dir_thr_120,81868,0.335077,0.277551,0.574517,0.697684,0.497021,0.335077,0.333333,0.001744,120,none,logistic_regression


In [ ]:
df_logistic_bal

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,logistic_regression_balanced,test,30,t2_dir_thr_120,93990,0.405681,0.402426,0.514961,0.543941,0.413386,0.405681,0.333333,0.072347,120,balanced,logistic_regression_balanced
1,logistic_regression_balanced,valid,30,t2_dir_thr_120,93508,0.390901,0.394348,0.637488,0.673857,0.421995,0.390901,0.333333,0.057568,120,balanced,logistic_regression_balanced
2,logistic_regression_balanced,test,30,t2_dir_thr_90,93990,0.409602,0.406155,0.517452,0.543388,0.415313,0.409602,0.333333,0.076269,90,balanced,logistic_regression_balanced
3,logistic_regression_balanced,valid,30,t2_dir_thr_90,93508,0.404235,0.409582,0.637965,0.667130,0.429376,0.404235,0.333333,0.070901,90,balanced,logistic_regression_balanced
4,logistic_regression_balanced,test,60,t2_dir_thr_120,88140,0.408355,0.404759,0.505053,0.530247,0.413424,0.408355,0.333333,0.075021,120,balanced,logistic_regression_balanced
5,logistic_regression_balanced,valid,60,t2_dir_thr_120,87688,0.392961,0.396523,0.620292,0.653533,0.420156,0.392961,0.333333,0.059628,120,balanced,logistic_regression_balanced
6,logistic_regression_balanced,test,60,t2_dir_thr_90,88140,0.409766,0.405476,0.502719,0.522646,0.412470,0.409766,0.333333,0.076433,90,balanced,logistic_regression_balanced
7,logistic_regression_balanced,valid,60,t2_dir_thr_90,87688,0.416490,0.420733,0.619749,0.635150,0.429252,0.416490,0.333333,0.083157,90,balanced,logistic_regression_balanced
8,logistic_regression_balanced,test,90,t2_dir_thr_120,82290,0.410049,0.406857,0.496935,0.521218,0.415131,0.410049,0.333333,0.076716,120,balanced,logistic_regression_balanced
9,logistic_regression_balanced,valid,90,t2_dir_thr_120,81868,0.393270,0.396271,0.613368,0.649546,0.423850,0.393270,0.333333,0.059937,120,balanced,logistic_regression_balanced
